# 사전확인

In [ ]:
single_dep = {f'{r.sigun}|{r.routeid}' for r in times.itertuples() if r.single}
print('단일 출발 운행:', len(single_dep), '/ 모델에 있는 것:', len(single_dep & set(nodes_all)))

N = 1000
rng = np.random.default_rng(1)
t0 = time.time()
for hr in (300, 500):
    for scen, wt in (('균등', False), ('단일출발3배', True)):
        for fr in FR:
            out = {}
            for sg in T:
                out.update(run_mc(sg, hr, fr, wt))
            mc[(hr, scen, fr)] = out
        print(f'병원{hr}m {scen} 완료 ({round(time.time()-t0)}초)')

In [ ]:
for hr in (300, 500):
    T1 = pd.DataFrame({f'{s[:2]}{int(f*100)}%': {f'{sg} {n}': round(100 * mc[(hr, s, f)][name2a[(sg, n)]][0])
                                                for sg, n in KEY}
                       for s in ('균등', '단일출발3배') for f in FR})
    print(f'\n=== 병원 {hr}m: 직통이 없어질 확률(%) N={N} ===')
    print(T1.to_string())

flat = [{'병원반경': hr, '시나리오': s, '감축비율': f, 'adm_cd8': a, '시군': sgn[a], '읍면동': nm[a],
         'P_직통없음': v[0], 'P_연결없음': v[1]} for (hr, s, f), m in mc.items() for a, v in m.items()]
pd.DataFrame(flat).to_csv('/content/outputs/montecarlo.csv', index=False, encoding='utf-8-sig')

# 방향 반영 직통

In [ ]:
def build_dir(df, merge_radius=60):
    g = gpd.GeoDataFrame(df.copy(), geometry=gpd.points_from_xy(df['lon'], df['lat']), crs='EPSG:4326').to_crs('EPSG:5179')
    g['x'], g['y'] = g.geometry.x, g.geometry.y
    s = g.drop_duplicates(['sigun', 'stop_id'])[['sigun', 'stop_id', 'stop_name', 'x', 'y']].reset_index(drop=True)
    s['nname'] = s['stop_name'].map(norm_name)
    parent = list(range(len(s)))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a
    for _, grp in s.groupby('sigun'):
        pts, names, ids = grp[['x', 'y']].to_numpy(), grp['nname'].to_numpy(), grp.index.to_numpy()
        for i, j in cKDTree(pts).query_pairs(merge_radius):
            if names[i] == names[j]:
                ra, rb = find(ids[i]), find(ids[j])
                if ra != rb:
                    parent[rb] = ra
    s['node'] = [f'{sg}:{find(i)}' for i, sg in zip(s.index, s['sigun'])]
    link = g[['sigun', 'line', 'stop_id', 'stop_seq']].merge(s[['sigun', 'stop_id', 'node']], on=['sigun', 'stop_id'], how='left')
    link['rkey'] = link['sigun'] + '|' + link['line']
    return link

link = build_dir(df2)
seq = {rk: list(dict.fromkeys(grp['node'])) for rk, grp in link.sort_values('stop_seq').groupby('rkey')}
print('정합성(기존 노선 집합과 같은 노선 수):', sum(set(v) == nodes_all.get(k) for k, v in seq.items()), '/', len(nodes_all))

In [ ]:
def truncated(sg, hn):
    """병원 정류장까지만 남김(병원을 지나 뒤에 오는 정류장은 병원행 승차 불가)"""
    out = {}
    for k, nodes in seq.items():
        if not k.startswith(sg + '|'):
            continue
        idx = [i for i, n in enumerate(nodes) if n in hn]
        out[k] = set(nodes[:idx[-1] + 1]) if idx else set(nodes)
    return out

rows, st = [], {}
for hr in (300, 500):
    for sg in T:
        e, rn, hn, nof = sg_inputs(sg, hr)
        nd = truncated(sg, hn)
        pop = e.set_index('adm_cd8')['pop65'].fillna(0).to_dict()
        base, imp = simulate_line_removal(rn, hn, nd, pop)
        base0 = level_for_regions(rn, hn, nof)
        c0, c1 = Counter(v[0] for v in base0.values()), Counter(v[0] for v in base.values())
        single = [a for a, (l, d) in base.items() if l == 0 and len(d) == 1]
        rows.append({'병원': hr, '시군': sg,
                     '방향무시(직/환/불)': f'{c0[0]}/{c0[1]}/{c0[9]}',
                     '방향반영(직/환/불)': f'{c1[0]}/{c1[1]}/{c1[9]}',
                     '단일의존': len(single), '단일의존_65+': int(sum(pop.get(a, 0) for a in single)),
                     '직통잃는운행': int((imp['n_lost_direct'] > 0).sum())})
        for a, (l, d) in base.items():
            st[(sg, nm[a], hr)] = ('단일' if (l == 0 and len(d) == 1) else f'직통({len(d)})' if l == 0 else '환승' if l == 1 else '불가/미확인')

print(pd.DataFrame(rows).to_string(index=False))
print('\n=== 핵심 8개 읍면, 방향 반영 ===')
print(pd.DataFrame({f'병원{hr}m': {f'{sg} {n}': st.get((sg, n, hr), '-') for sg, n in KEY} for hr in (300, 500)}).to_string())

# 무작위 감축

In [ ]:
 nd_cache = {(sg, hr): truncated(sg, sg_inputs(sg, hr)[2]) for sg in T for hr in (300, 500)}

def run_mc_dir(sg, hr, frac, weighted):
    e, rn, hn, nof = sg_inputs(sg, hr)
    nd = nd_cache[(sg, hr)]
    keys = list(nd)
    k = max(1, round(len(keys) * frac))
    p = None
    if weighted:
        w = np.array([3.0 if x in single_dep else 1.0 for x in keys]); p = w / w.sum()
    cnt, cl = Counter(), Counter()
    for _ in range(N):
        idx = rng.choice(len(keys), size=k, replace=False, p=p)
        drop = {keys[i] for i in idx}
        lv = level_for_regions(rn, hn, {x: v for x, v in nd.items() if x not in drop})
        for a, (l, _) in lv.items():
            if l > 0: cnt[a] += 1
            if l == 9: cl[a] += 1
    return {a: (cnt[a] / N, cl[a] / N) for a in rn}

base_dir = {}
for hr in (300, 500):
    for sg in T:
        e, rn, hn, nof = sg_inputs(sg, hr)
        for a, (l, d) in level_for_regions(rn, hn, nd_cache[(sg, hr)]).items():
            base_dir[(hr, a)] = (l, len(d))

N = 1000
rng = np.random.default_rng(2)
KEY2 = KEY + [('공주시', '반포면')]
mcd, t0 = {}, time.time()
for hr in (300, 500):
    for scen, wt in (('균등', False), ('단일출발3배', True)):
        for fr in FR:
            out = {}
            for sg in T:
                out.update(run_mc_dir(sg, hr, fr, wt))
            mcd[(hr, scen, fr)] = out
        print(f'병원{hr}m {scen} 완료 ({round(time.time()-t0)}초)')

LB = {0: '직통', 1: '환승', 9: '불가'}
for hr in (300, 500):
    T1 = pd.DataFrame({f'{s[:2]}{int(f*100)}%': {f'{sg} {n}': round(100 * mcd[(hr, s, f)][name2a[(sg, n)]][0])
                                                for sg, n in KEY2}
                       for s in ('균등', '단일출발3배') for f in FR})
    print(f'\n=== 방향 반영, 병원 {hr}m: 직통이 없어질 확률(%) ===')
    print(T1.to_string())

for hr in (300, 500):
    m = mcd[(hr, '균등', 0.2)]
    d = pd.DataFrame([{'읍면동': f'{sgn[a]} {nm[a]}',
                       '기본연결': LB[base_dir[(hr, a)][0]] + (f'({base_dir[(hr, a)][1]})' if base_dir[(hr, a)][0] == 0 else ''),
                       '직통없음%': round(100 * v[0]), '연결없음%': round(100 * v[1]), '65세이상': int(p65[a])}
                      for a, v in m.items()])
    print(f'\n=== 병원 {hr}m, 균등 20%: 직통 없음 확률 상위 12 ===')
    print(d.sort_values(['직통없음%', '65세이상'], ascending=False).head(12).to_string(index=False))

print('\n=== 현재 직통이 있는 읍면 중 20% 감축 시 직통을 잃는 65세 이상 기대 인구 ===')
for hr in (300, 500):
    for s in ('균등', '단일출발3배'):
        m = mcd[(hr, s, 0.2)]
        tot = Counter()
        for a, (pn, pl) in m.items():
            if base_dir[(hr, a)][0] == 0:
                tot[sgn[a]] += pn * p65[a]
        print(f'병원{hr}m {s}:', {k: round(v) for k, v in tot.items()})

flat = [{'병원반경': hr, '시나리오': s, '감축비율': f, 'adm_cd8': a, '시군': sgn[a], '읍면동': nm[a],
         'P_직통없음': v[0], 'P_연결없음': v[1]} for (hr, s, f), m in mcd.items() for a, v in m.items()]
pd.DataFrame(flat).to_csv('/content/outputs/montecarlo_directional.csv', index=False, encoding='utf-8-sig')

# 반포면, 조마/감천면 직통 운행 확인

In [ ]:
tm = times.drop_duplicates(['sigun', 'routeid']).set_index(['sigun', 'routeid'])
def hhmm(v):
    return '-' if pd.isna(v) else f'{int(v // 60):02d}:{int(v % 60):02d}'

rowsR = []
for sg, n in [('공주시', '반포면'), ('김천시', '조마면'), ('김천시', '감천면')]:
    a = name2a[(sg, n)]
    for hr in (300, 500):
        e, rn, hn, nof = sg_inputs(sg, hr)
        nd = nd_cache[(sg, hr)]
        lv, d = level_for_regions(rn, hn, nd)[a]
        for k in sorted(d):
            no, s0, s1, _, _, ns = route_info(k)
            rid = k.split('|', 1)[1]
            f_, l_ = (tm.loc[(sg, rid), 'f'], tm.loc[(sg, rid), 'l']) if (sg, rid) in tm.index else (np.nan, np.nan)
            dist = hdist(sg, nodes_all[k] & hn)
            rowsR.append({'읍면': f'{sg} {n}', '병원반경': hr, '직통수': len(d), 'route_id': rid, '번호': no,
                          '출발→종점': f'{s0}→{s1}', '첫차': hhmm(f_), '막차': hhmm(l_), '정류장수': ns,
                          '병원까지(m)': None if dist is None else round(dist)})

In [ ]:
R = pd.DataFrame(rowsR)
print(R[R['병원반경'] == 300].to_string(index=False))
print('\n500m에서 직통 수 요약:')
print(R[R['병원반경'] == 500].groupby('읍면').agg(직통수=('직통수', 'first')).to_string())

# 이미지3

In [ ]:
from shapely.ops import unary_union
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

In [ ]:
# ---------- 방향 반영 유형 (병원 300m) ----------
R2 = (sub[['adm_cd8', '시군구명칭', '읍면동명칭', 'pop65', 'ratio65', 'd_near_m', 'hosp_km']]
      .rename(columns={'시군구명칭': 'sigun', '읍면동명칭': '읍면동'}).copy())
rn300 = setup[300][0]
R2['n_nodes_in'] = [len(rn300.get(s, {}).get(a, set())) for s, a in zip(R2['sigun'], R2['adm_cd8'])]
R2['lv'] = [base_dir[(300, a)][0] for a in R2['adm_cd8']]
R2['nd'] = [base_dir[(300, a)][1] for a in R2['adm_cd8']]
R2['lv5'] = [base_dir[(500, a)][0] for a in R2['adm_cd8']]
R2['nd5'] = [base_dir[(500, a)][1] for a in R2['adm_cd8']]

def cat3(r):
    if r['n_nodes_in'] == 0: return '노선 데이터 미확인'
    if r['lv'] == 1: return '환승 필요'
    if r['lv'] == 0 and r['nd'] == 1: return '단일 노선 의존'
    return '그 외'
R2['cat'] = R2.apply(cat3, axis=1)
R2['dag'] = R2['cat'].isin(['단일 노선 의존', '환승 필요']) & (R2['lv5'] == 0) & (R2['nd5'] >= 2)
print('=== 방향 반영 유형별 읍면동 수 (병원 300m) ===')
print(R2.groupby(['sigun', 'cat']).size().unstack(fill_value=0).to_string())

In [ ]:
if 'cover500' not in sub.columns:
    sub['cover500'] = sub.geometry.intersection(unary_union(stg.geometry.buffer(500))).area / sub.geometry.area
c = R2.merge(sub[['adm_cd8', 'cover500']], on='adm_cd8')
c['r_near'] = c['d_near_m'].rank(ascending=False, method='min'); c['r_cov'] = c['cover500'].rank(method='min')
c['r_hosp'] = c['hosp_km'].rank(ascending=False, method='min'); c['r_old'] = c['ratio65'].rank(ascending=False, method='min')
V = c[c['cat'].isin(['단일 노선 의존', '환승 필요'])]
print('\n=== 후보와 기존 지표 순위 (52개 중, 1=가장 취약) ===')
print(V[['sigun', '읍면동', 'pop65', 'cat', 'r_near', 'r_cov', 'r_hosp', 'r_old']].round(0).to_string(index=False))
for k in (5, 10, 15, 20):
    o = {nm_: f"{int((V[col] <= k).sum())} ({len(V) * k / len(c):.1f})" for col, nm_ in
         [('r_near', '정류장거리'), ('r_cov', '커버율'), ('r_hosp', '병원거리'), ('r_old', '고령비율')]}
    a3 = int(((V['r_near'] <= k) | (V['r_cov'] <= k) | (V['r_hosp'] <= k)).sum())
    print(f'k={k}:', o, f'| 거리계 3개 중 하나라도 상위k: {a3}/{len(V)}')

In [ ]:
S = sub.merge(R2[['adm_cd8', 'cat', 'dag']], on='adm_cd8')
fig, axs = plt.subplots(1, 3, figsize=(13, 5.8))
for ax, sg in zip(axs, T):
    d = S[S['시군구명칭'] == sg]
    d.plot(ax=ax, color=d['cat'].map(COL).tolist(), edgecolor='white', linewidth=0.8)
    na = d[d['cat'] == '노선 데이터 미확인']
    if len(na): na.plot(ax=ax, color=C_NA, edgecolor=INK2, linewidth=0.8, hatch='///')
    hg[hg['시군구'] == sg].plot(ax=ax, color=INK, marker='P', markersize=50, zorder=5)
    for _, r in d[d['cat'] != '그 외'].iterrows():
        p = r.geometry.representative_point()
        ax.annotate(r['읍면동명칭'] + ('†' if r['dag'] else ''), (p.x, p.y), ha='center', va='center', fontsize=8.5,
                    color=INK, zorder=6, bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.85))
    if (d['cat'] != '그 외').sum() == 0:
        ax.text(0.5, -0.01, '해당 읍면 없음', transform=ax.transAxes, ha='center', va='top', fontsize=9.5, color=INK2)
    ax.set_title(sg, fontsize=12, color=INK, y=1.0); ax.set_axis_off()
handles = [Patch(fc=C_SINGLE, label='병원행 직통이 운행 1개뿐'), Patch(fc=C_TRANS, label='병원행 직통 없음(환승 필요)'),
           Patch(fc=C_NA, ec=INK2, hatch='///', label='노선 데이터 미확인(아포읍)'),
           Patch(fc=C_OTHER, label='그 외 (병원행 직통 2개 이상)'),
           Line2D([], [], marker='P', color='w', markerfacecolor=INK, markersize=9, label='병원(병원·종합병원)')]
fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 0.09), ncol=3, frameon=False, fontsize=9)
fig.text(0.01, 0.05, '병원행 방향 직통만 인정(병원을 지난 뒤 정류장 제외), 병원 반경 300m, 개별 운행(route_id) 단위. 환승은 방향 미반영(낙관적).', fontsize=7.5, color=INK2)
fig.text(0.01, 0.015, '† 병원 반경을 500m로 넓히면 터미널행 등 직통이 다수 있으나 병원까지 약 330~430m를 걸어야 함. 경계 2025-06-30, 정류장 2025-10-31, 병원 2026-10-12, 인구 2026-09-30.', fontsize=7.5, color=INK2)
fig.tight_layout(rect=(0, 0.17, 1, 1))
fig.savefig('/content/outputs/fig1_map.png', bbox_inches='tight', facecolor='white'); plt.show()

In [ ]:
rowsF = []
for a in sub['adm_cd8']:
    if base_dir[(300, a)][0] != 0: continue
    ps = [mcd[(300, '균등', f)][a][0] for f in FR]
    if max(ps) >= 0.05:
        rowsF.append((f"{sgn[a][:2]} {nm[a]}\n65세 이상 {int(p65[a]):,}명", ps))
rowsF.sort(key=lambda t: -t[1][1])
fig, ax = plt.subplots(figsize=(9, 1.8 + 1.1 * len(rowsF)))
H, AL = 0.22, (0.4, 0.7, 1.0)
for i, (lab, ps) in enumerate(rowsF):
    for j, (p_, f) in enumerate(zip(ps, FR)):
        y = i + (j - 1) * H * 1.1
        ax.barh(y, 100 * p_, height=H, color=C_SINGLE, alpha=AL[j], edgecolor='white')
        ax.text(100 * p_ + 0.8, y, f'{100 * p_:.0f}%', va='center', fontsize=8.5, color=INK)
ax.set_yticks(range(len(rowsF))); ax.set_yticklabels([r[0] for r in rowsF], fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('병원행 직통이 없어질 확률(%)', color=INK2)
ax.set_xlim(0, 100 * max(max(r[1]) for r in rowsF) * 1.2)
ax.legend(handles=[Patch(fc=C_SINGLE, alpha=a_, label=f'운행 {int(f * 100)}% 무작위 감축') for a_, f in zip(AL, FR)],
          frameon=False, loc='lower right', fontsize=9)
fig.text(0.02, -0.03, '병원 반경 300m, 방향 반영, 무작위 감축 1,000회(균등). 이미 직통이 없는 읍면(환승 필요 6곳)과 아포읍은 제외.\n병원 반경 500m에서는 공주 반포면만 해당(11/19/30%). 실제 폐지 예측이 아니며 시간·배차는 반영하지 않음.', fontsize=8, color=INK2)
fig.savefig('/content/outputs/fig2_montecarlo.png', bbox_inches='tight', facecolor='white'); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 6))
g0 = R2[R2['cat'] == '그 외']
ax.scatter(g0['d_near_m'], g0['hosp_km'], s=28, color='#b9b8b2', edgecolor='white', linewidth=0.8, zorder=2)
for k in ('단일 노선 의존', '환승 필요'):
    g1 = R2[R2['cat'] == k]
    ax.scatter(g1['d_near_m'], g1['hosp_km'], s=70, color=COL[k], edgecolor='white', linewidth=1.2, zorder=3, label=k)
na = R2[R2['cat'] == '노선 데이터 미확인']
ax.scatter(na['d_near_m'], na['hosp_km'], s=70, facecolor='white', edgecolor=INK2, linewidth=1.5, zorder=3, label='노선 데이터 미확인')
OFF = {'탄천면': (10, 4, 'left'), '구성면': (9, -13, 'left'), '지례면': (9, 6, 'left'), '부항면': (-9, 7, 'right'),
       '대덕면': (-9, 8, 'right'), '증산면': (-9, -15, 'right'), '아포읍': (-9, 9, 'right'),
       '감천면': (9, 6, 'left'), '조마면': (9, -13, 'left'), '반포면': (9, 6, 'left')}
for _, r in R2[R2['cat'] != '그 외'].iterrows():
    dx, dy, ha = OFF.get(r['읍면동'], (8, 6, 'left'))
    ax.annotate(f"{r['sigun'][:2]} {r['읍면동']}", (r['d_near_m'], r['hosp_km']), xytext=(dx, dy),
                textcoords='offset points', fontsize=9, color=INK, ha=ha)
ax.axvline(R2['d_near_m'].median(), color='#d4d3ce', lw=1, ls='--', zorder=1)
ax.axhline(R2['hosp_km'].median(), color='#d4d3ce', lw=1, ls='--', zorder=1)
ax.set_ylim(-1, 31)
ax.set_xlabel('가장 가까운 정류장까지 거리(m, 읍면동 대표점)', color=INK2); ax.set_ylabel('같은 시군 병원까지 직선거리(km)', color=INK2)
ax.legend(frameon=False, loc='lower right', fontsize=9)
ax.text(0.01, -0.12, '점선은 52개 읍면동 중앙값. 회색 점은 나머지 읍면동. 유형은 방향 반영·병원 반경 300m 기준.', transform=ax.transAxes, fontsize=8, color=INK2)
fig.savefig('/content/outputs/fig3_stop_vs_hospital.png', bbox_inches='tight', facecolor='white'); plt.show()

- 산점도 수정

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 6))
g0 = R2[R2['cat'] == '그 외']
ax.scatter(g0['d_near_m'], g0['hosp_km'], s=28, color='#b9b8b2', edgecolor='white', linewidth=0.8, zorder=2)
for k in ('단일 노선 의존', '환승 필요'):
    g1 = R2[R2['cat'] == k]
    ax.scatter(g1['d_near_m'], g1['hosp_km'], s=70, color=COL[k], edgecolor='white', linewidth=1.2, zorder=3, label=k)
na = R2[R2['cat'] == '노선 데이터 미확인']
ax.scatter(na['d_near_m'], na['hosp_km'], s=70, facecolor='white', edgecolor=INK2, linewidth=1.5, zorder=3, label='노선 데이터 미확인')
OFF = {'탄천면': (10, 4, 'left'), '구성면': (9, -13, 'left'), '지례면': (9, 6, 'left'), '부항면': (-9, 7, 'right'),
       '대덕면': (-9, 8, 'right'), '증산면': (-9, -15, 'right'), '아포읍': (0, -19, 'center'),
       '감천면': (9, 6, 'left'), '조마면': (9, 8, 'left'), '반포면': (9, 6, 'left')}
for _, r in R2[R2['cat'] != '그 외'].iterrows():
    dx, dy, ha = OFF.get(r['읍면동'], (8, 6, 'left'))
    ax.annotate(f"{r['sigun'][:2]} {r['읍면동']}", (r['d_near_m'], r['hosp_km']), xytext=(dx, dy),
                textcoords='offset points', fontsize=9, color=INK, ha=ha)
ax.axvline(R2['d_near_m'].median(), color='#d4d3ce', lw=1, ls='--', zorder=1)
ax.axhline(R2['hosp_km'].median(), color='#d4d3ce', lw=1, ls='--', zorder=1)
ax.set_ylim(-1, 31)
ax.set_xlabel('가장 가까운 정류장까지 거리(m, 읍면동 대표점)', color=INK2); ax.set_ylabel('같은 시군 병원까지 직선거리(km)', color=INK2)
ax.legend(frameon=False, loc='lower right', fontsize=9)
ax.text(0.01, -0.12, '점선은 52개 읍면동 중앙값. 회색 점은 나머지 읍면동. 유형은 방향 반영·병원 반경 300m 기준.', transform=ax.transAxes, fontsize=8, color=INK2)
fig.savefig('/content/outputs/fig3_stop_vs_hospital.png', bbox_inches='tight', facecolor='white'); plt.show()

from google.colab import files
files.download('/content/outputs/fig3_stop_vs_hospital.png')

# 방향 반영 6개 조합 확인

In [ ]:
def make_seq(mr):
    link = build_dir(df2, mr)
    return {rk: list(dict.fromkeys(g['node'])) for rk, g in link.sort_values('stop_seq').groupby('rkey')}

def trunc(seq_, sg, hn):
    out = {}
    for k, nodes in seq_.items():
        if not k.startswith(sg + '|'):
            continue
        idx = [i for i, n in enumerate(nodes) if n in hn]
        out[k] = set(nodes[:idx[-1] + 1]) if idx else set(nodes)
    return out

CAND = [('공주시', '탄천면'), ('공주시', '반포면'), ('김천시', '조마면'), ('김천시', '감천면'),
        ('김천시', '구성면'), ('김천시', '지례면'), ('김천시', '부항면'), ('김천시', '대덕면'), ('김천시', '증산면')]

In [ ]:
rowsD, stD = [], {}

In [ ]:
for mr in (30, 60, 100):
    node_xy_m, nodes_all_m = build_network(df2, mr)
    seq_m = make_seq(mr)
    ok = sum(set(v) == nodes_all_m.get(k) for k, v in seq_m.items())
    print(f'병합{mr}m 노선 집합 정합성: {ok} / {len(nodes_all_m)}')
    for hr in (300, 500):
        rn_all, hn_all = assign_nodes(node_xy_m, sub, hg, hr)
        tag = f'병합{mr}·병원{hr}'
        for sg in T:
            e = sub[sub['시군구명칭'] == sg]
            rn = {a: rn_all.get(sg, {}).get(a, set()) for a in e['adm_cd8']}
            hn = hn_all.get(sg, set())
            nd = trunc(seq_m, sg, hn)
            pop = e.set_index('adm_cd8')['pop65'].fillna(0).to_dict()
            base, imp = simulate_line_removal(rn, hn, nd, pop)
            c1 = Counter(v[0] for v in base.values())
            single = [a for a, (l, d) in base.items() if l == 0 and len(d) == 1]
            rowsD.append({'설정': tag, '시군': sg, '직통': c1[0], '환승1': c1[1], '불가': c1[9],
                          '단일의존': len(single), '단일의존_65+': int(sum(pop.get(a, 0) for a in single)),
                          '직통잃는운행': int((imp['n_lost_direct'] > 0).sum())})
            for a, (l, d) in base.items():
                stD[(sg, nm[a], tag)] = ('단일' if (l == 0 and len(d) == 1) else f'직통({len(d)})' if l == 0
                                         else '환승' if l == 1 else '불가/미확인')

In [ ]:
print(pd.DataFrame(rowsD).to_string(index=False))
print('\n=== 후보 9곳 상태 (방향 반영) ===')
tags = [f'병합{mr}·병원{hr}' for mr in (30, 60, 100) for hr in (300, 500)]
print(pd.DataFrame({t: {f'{sg} {n}': stD.get((sg, n, t), '-') for sg, n in CAND} for t in tags}).to_string())

# 수치 대조

In [ ]:
print('정류장 거리 평균/중앙값:', round(sub['d_near_m'].mean(), 1), sub['d_near_m'].median())
print('병원 거리 평균/최대(km):', round(sub['hosp_km'].mean(), 1), round(sub['hosp_km'].max(), 1))
print(sub[['hosp_km', 'ratio65', 'd_near_m']].corr(method='spearman').round(2))
a = set(sub.nlargest(10, 'd_near_m')['adm_cd8']); b = set(sub.nlargest(10, 'hosp_km')['adm_cd8'])
print('정류장·병원 거리 상위 10 겹침:', len(a & b), '(우연 기대', round(10 * 10 / 52, 1), ')')
print(sub.loc[sub['읍면동명칭'].isin(['탄천면', '해리면']), ['시군구명칭', '읍면동명칭', 'd_near_m', 'hosp_km']].round(1).to_string(index=False))